In [1]:
from typing import Annotated, Sequence, TypedDict 
from langchain_core.messages import BaseMessage # The foundational class for all message types in LangGraph
from langchain_core.messages import ToolMessage # Passes data back to LLM after it calls a tool such as the content and the tool_call_id
from langchain_core.messages import SystemMessage # Message for providing instructions to the LLM
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

GOOGLE_API_KEY = "***"

In [23]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [24]:
@tool
def add(a:int, b:int):
    """This is an addition function that sums up 2 numbers"""
    return a+b

@tool
def subtract(a: int, b: int):
    """Subtraction function"""
    return a - b

@tool
def multiply(a: int, b: int):
    """Multiplication function"""
    return a * b

tools = [add, subtract, multiply]

In [25]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=GOOGLE_API_KEY).bind_tools(tools)

In [26]:
def model_call(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content="You are my AI assistant. Please answer my queries to the best of your ability.")
    response = model.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}

In [27]:
def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

In [28]:
graph = StateGraph(AgentState)
graph.add_node("our_agent", model_call)


tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)

graph.set_entry_point("our_agent")

graph.add_conditional_edges(
    "our_agent",
    should_continue,
    {
        "continue": "tools",
        "end": END
    }
)

graph.add_edge("tools", "our_agent")

app = graph.compile()

In [29]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

In [30]:
inputs = {"messages": [("user", "Multiply 20 and 6. Subtract 51.")]}
print_stream(app.stream(inputs, stream_mode="values"))

================================ Human Message =================================

Multiply 20 and 6. Subtract 51.
================================== Ai Message ==================================
Tool Calls:
  multiply (f7c3f1a1-a926-41d6-a13e-3a77251de137)
 Call ID: f7c3f1a1-a926-41d6-a13e-3a77251de137
  Args:
    a: 20
    b: 6
================================= Tool Message =================================
Name: multiply

120
================================== Ai Message ==================================
Tool Calls:
  subtract (44522d76-dab5-4893-854c-866b1b61bbf7)
 Call ID: 44522d76-dab5-4893-854c-866b1b61bbf7
  Args:
    b: 51
    a: 120
================================= Tool Message =================================
Name: subtract

69
================================== Ai Message ==================================

The answer is 69.
